In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
import wandb
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# -----------------------------
# wandb init
# -----------------------------
wandb.init(
    project="brain-tumor-autoencoder-anomaly",
    config={
        "latent_dim": 128,
        "batch_size": 64,
        "lr_ae": 1e-3,
        "epochs_ae": 25,
        "img_size": 128,
        "threshold_percentile": 95,

        # NEW hyperparameters
        "base_channels": 32,
        "num_blocks": 3,
        "kernel_size": 3,
        "activation": "leakyrelu",
        "norm": "batch",
        "dropout": 0.1,
        "output_activation": "tanh"
    }
)
config = wandb.config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Dataset
# -----------------------------
dataset = load_dataset("PranomVignesh/MRI-Images-Of-Brain-Tumor")

transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

class HFBrainDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]["image"].convert("L")
        label = self.data[idx]["label"]
        if self.transform:
            img = self.transform(img)
        return img, label

def filter_no_tumor(hf_split):
    return [item for item in hf_split if item["label"] == 2]

train_no_tumor = filter_no_tumor(dataset["train"])
val_no_tumor   = filter_no_tumor(dataset["validation"])

train_ds = HFBrainDataset(train_no_tumor, transform)
val_ds   = HFBrainDataset(val_no_tumor, transform)
test_ds  = HFBrainDataset(dataset["test"], transform)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# -----------------------------
# Flexible Autoencoder
# -----------------------------
class FlexibleAutoencoder(nn.Module):
    def __init__(
        self,
        img_channels=1,
        base_channels=32,
        num_blocks=3,
        latent_dim=128,
        kernel_size=3,
        activation="relu",
        norm="batch",
        dropout=0.0,
        output_activation="tanh"
    ):
        super().__init__()

        # Activation
        act = {
            "relu": nn.ReLU,
            "leakyrelu": lambda: nn.LeakyReLU(0.2),
            "elu": nn.ELU,
            "gelu": nn.GELU
        }[activation]

        # Normalization
        def norm_layer(c):
            if norm == "batch":
                return nn.BatchNorm2d(c)
            elif norm == "layer":
                return nn.GroupNorm(1, c)
            else:
                return nn.Identity()

        # Encoder
        enc_layers = []
        in_c = img_channels
        channels = []

        for i in range(num_blocks):
            out_c = base_channels * (2 ** i)
            channels.append(out_c)

            enc_layers += [
                nn.Conv2d(in_c, out_c, kernel_size, stride=2, padding=1),
                norm_layer(out_c),
                act(),
            ]

            if dropout > 0:
                enc_layers.append(nn.Dropout2d(dropout))

            in_c = out_c

        self.encoder = nn.Sequential(*enc_layers)

        # compute flattened size
        dummy = torch.zeros(1, img_channels, 128, 128)
        with torch.no_grad():
            enc_out = self.encoder(dummy)
        self.flat_dim = enc_out.numel()

        self.enc_fc = nn.Linear(self.flat_dim, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, self.flat_dim)

        # Decoder
        dec_layers = []
        channels.reverse()

        for i in range(num_blocks):
            out_c = channels[i] // 2 if i < num_blocks - 1 else img_channels

            dec_layers += [
                nn.ConvTranspose2d(
                    channels[i],
                    out_c,
                    kernel_size,
                    stride=2,
                    padding=1,
                    output_padding=1
                )
            ]

            if i < num_blocks - 1:
                dec_layers += [
                    norm_layer(out_c),
                    act()
                ]
                if dropout > 0:
                    dec_layers.append(nn.Dropout2d(dropout))

        self.decoder = nn.Sequential(*dec_layers)

        if output_activation == "tanh":
            self.out_act = nn.Tanh()
        elif output_activation == "sigmoid":
            self.out_act = nn.Sigmoid()
        else:
            self.out_act = nn.Identity()

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.enc_fc(x)

    def decode(self, z):
        x = self.dec_fc(z)
        x = x.view(-1, self.encoder[-3].num_features, 16, 16)
        x = self.decoder(x)
        return self.out_act(x)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

# Instantiate model
ae = FlexibleAutoencoder(
    img_channels=1,
    base_channels=config.base_channels,
    num_blocks=config.num_blocks,
    latent_dim=config.latent_dim,
    kernel_size=config.kernel_size,
    activation=config.activation,
    norm=config.norm,
    dropout=config.dropout,
    output_activation=config.output_activation
).to(device)

criterion_rec = nn.MSELoss()
optimizer_ae = torch.optim.Adam(ae.parameters(), lr=config.lr_ae)

# -----------------------------
# Reconstruction error helper
# -----------------------------
def reconstruction_errors(model, loader):
    model.eval()
    errors = []
    labels = []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            recon, _ = model(imgs)
            err = torch.mean((recon - imgs) ** 2, dim=[1, 2, 3]).cpu().numpy()
            errors.extend(err)
            labels.extend(lbls.numpy())
    return np.array(errors), np.array(labels)

# -----------------------------
# Train Autoencoder
# -----------------------------
for epoch in range(config.epochs_ae):
    ae.train()
    total_loss = 0

    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        optimizer_ae.zero_grad()
        recon, _ = ae(imgs)
        loss = criterion_rec(recon, imgs)
        loss.backward()
        optimizer_ae.step()
        total_loss += loss.item() * imgs.size(0)

    train_loss = total_loss / len(train_ds)
    val_errors, _ = reconstruction_errors(ae, val_loader)
    val_loss = val_errors.mean()

    wandb.log({
        "AE_train_loss": train_loss,
        "AE_val_loss": val_loss,
        "epoch": epoch
    })

    print(f"Epoch {epoch+1}/{config.epochs_ae} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# -----------------------------
# Threshold selection
# -----------------------------
val_errors, _ = reconstruction_errors(ae, val_loader)
threshold = np.percentile(val_errors, config.threshold_percentile)
wandb.log({"threshold": threshold})
print(f"Chosen threshold: {threshold:.6f}")

# -----------------------------
# Evaluate on test set
# -----------------------------
test_errors, test_labels = reconstruction_errors(ae, test_loader)

preds = (test_errors > threshold).astype(int)
true = (test_labels != 2).astype(int)

cm = confusion_matrix(true, preds)
print("Confusion matrix:")
print(cm)

wandb.log({
    "anomaly_confusion_matrix": wandb.plot.confusion_matrix(
        y_true=true,
        preds=preds,
        class_names=["no_tumor", "tumor"]
    )
})

# -----------------------------
# Visualize detected tumors
# -----------------------------
def show_detected_tumors(model, loader, threshold, max_images=5):
    model.eval()
    shown = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        recon, _ = model(imgs)

        pixel_err = (recon - imgs) ** 2
        img_err = pixel_err.mean(dim=[1,2,3]).detach().cpu().numpy()

        for i in range(len(imgs)):
            if img_err[i] > threshold:
                orig = imgs[i].detach().cpu().squeeze()
                rec  = recon[i].detach().cpu().squeeze()
                heat = pixel_err[i].detach().cpu().squeeze()

                heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)

                fig, ax = plt.subplots(1, 3, figsize=(12,4))
                ax[0].imshow(orig, cmap="gray")
                ax[0].set_title("Original")

                ax[1].imshow(rec, cmap="gray")
                ax[1].set_title("Reconstruction")

                ax[2].imshow(orig, cmap="gray")
                ax[2].imshow(heat, cmap="jet", alpha=0.5)
                ax[2].set_title("Anomaly Heatmap")

                for a in ax:
                    a.axis("off")

                plt.show()

                wandb.log({
                    "Detected_Tumor": [
                        wandb.Image(orig, caption="Original"),
                        wandb.Image(rec, caption="Reconstruction"),
                        wandb.Image(heat, caption="Heatmap")
                    ]
                })

                shown += 1
                if shown >= max_images:
                    return

# Show 10 detected tumors
show_detected_tumors(ae, test_loader, threshold, max_images=10)

# -----------------------------
# Save model
# -----------------------------
torch.save(ae.state_dict(), "autoencoder_flexible.pt")
wandb.save("autoencoder_flexible.pt")

print("Done.")
